In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class DoubleConv(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.c1 = nn.Conv2d(in_c, out_c, 3, padding=0, bias=True)
        self.c2 = nn.Conv2d(out_c, out_c, 3, padding=0, bias=True)
        self.r = nn.ReLU(inplace=True)

    def forward(self, x):
        x = self.c1(x)
        x = self.c2(x)
        out = self.r(x)
        return out

class Down(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.pool = nn.MaxPool2d(2)
        self.block = DoubleConv(in_c, out_c)

    def forward(self, x):
        x = self.pool(x)
        out = self.block(x)
        return out

class Up(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_c, in_c // 2, 2, stride=2)
        self.block = DoubleConv(in_c, out_c)

    def forward(self, x, skip):
        x = self.up(x)
        th, tw = x.shape[-2:]
        Hs, Ws = skip.shape[-2:]
        dh, dw = (Hs - th)//2, (Ws - tw)//2
        skip_c = skip[...,dh:dh+th, dw:dw+tw]
        x = torch.cat([skip_c, x], dim=1)
        out = self.block(x)
        return out

class UNet(nn.Module):
    def __init__(self, in_c=3, n_classes=1, base=64):
        super().__init__()
        self.in_c = DoubleConv(in_c, base)
        self.down1 = Down(base, base * 2)
        self.down2 = Down(base * 2, base * 4)
        self.down3 = Down(base * 4, base * 8)
        self.down4 = Down(base * 8, base * 16)
        self.up1 = Up(base * 16, base * 8)
        self.up2 = Up(base * 8, base * 4)
        self.up3 = Up(base * 4, base * 2)
        self.up4 = Up(base * 2, base)
        self.outc = nn.Conv2d(base, n_classes, 1)

    def forward(self, x):
        # 인코딩 층
        x1 = self.in_c(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)

        # 디코더 층
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)

        # 출력 층
        out = self.outc(x)
        return out

In [2]:
#작업 준비
import os, time
from pathlib import Path
import numpy as np
from PIL import Image

from torch.utils.data import Dataset,DataLoader
from torchvision.transforms import functional as TF

import torch
import torch.nn as nn
import torch.nn.functional as F

import matplotlib.pyplot as plt

#상수정의
DEVICE = torch.device('mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu')
NUM_WORKERS = 4
DATA_ROOT='../../data/VOCdevkit'
SPLIT='VOC2012'
BATCH_SIZE=8
EPOCHS=50
LR=0.005
MOMENTUM=0.9
WEIGHT_DECAY=0.0001

IMG_SIZE=512
N_CLASSES=21
IGNORE_INDEX=255

OUT_DIR='runs/fcn_scratch'
BEST_PATH=os.path.join(OUT_DIR,'best_m.pth')
os.makedirs(OUT_DIR,exist_ok=True)

In [3]:
#데이터 수집(데이터 로드)
def make_transform(train):
    def _tfm(img,mask):
        img= TF.resize(img,(IMG_SIZE,IMG_SIZE))
        mask= TF.resize(mask,(IMG_SIZE,IMG_SIZE),interpolation=TF.InterpolationMode.NEAREST)
        if train and torch.rand(1)<0.5:
            img = TF.hflip(img)
            mask = TF.hflip(mask)
        img = TF.to_tensor(img)
        img = TF.normalize(img, [0.485,0.456,0.406], [0.229,0.224,0.225])
        mask=torch.as_tensor(np.array(mask),dtype=torch.long)
        return img,mask
    return _tfm
class VOCDatasetTrain(Dataset):
    def __init__(self,root,split,image_set,transforms=None):
        base=Path(root)/split
        self.img_dir=base/'JPEGImages'
        self.mask_dir=base/'SegmentationClass'
        with open(base/'ImageSets'/'Segmentation'/f'{image_set}.txt') as f:
            self.ids= [x.strip() for x in f if x.strip()]
        self.tfm=transforms
    def __len__(self):
        return len(self.ids)
    def __getitem__(self,idx):
        id_ =self.ids[idx]
        img=Image.open(self.img_dir/f'{id_}.jpg').convert('RGB')
        mask=Image.open(self.mask_dir/f'{id_}.png')
        if self.tfm:
            img,mask=self.tfm(img,mask)
        return img,mask

class VOCDatasetVal(VOCDatasetTrain):
    def __getitem__(self,idx):
        id_ =self.ids[idx]
        img=Image.open(self.img_dir/f'{id_}.jpg').convert('RGB')
        mask=Image.open(self.mask_dir/f'{id_}.png')
        if self.tfm:
            img,mask=self.tfm(img,mask)
        return img,mask,id_

def get_loaders():
    tr_ds=VOCDatasetTrain(DATA_ROOT,SPLIT,'train',transforms=make_transform(train=True))
    val_ds=VOCDatasetVal(DATA_ROOT,SPLIT,'val',transforms=make_transform(train=False))

    # mac
    tr_ds_loader = DataLoader(tr_ds,batch_size=BATCH_SIZE,shuffle=True)
    val_ds_loader = DataLoader(val_ds,batch_size=BATCH_SIZE,shuffle=False)
    # cuda
    # tr_ds_loader = DataLoader(tr_ds,batch_size=BATCH_SIZE,shuffle=True,num_workers=NUM_WORKERS, pin_memory=True)
    # val_ds_loader = DataLoader(val_ds,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS, pin_memory=True)
    return tr_ds,val_ds,tr_ds_loader,val_ds_loader

In [4]:
# 데이터 준비
tr_ds, val_ds, tr_ds_loader, val_ds_loader = get_loaders()

In [5]:
# 모델 학습
m = UNet(3, N_CLASSES).to(DEVICE)
criterion =nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)
opt = torch.optim.SGD(m.parameters(), lr=LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
# cuda
# scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())
# mac
scaler = torch.amp.GradScaler(enabled=torch.backends.mps.is_available())

/Users/ihanjo/Library/Python/3.13/lib/python/site-packages/torch/amp/grad_scaler.py:132: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  warnings.warn(


In [6]:
def center_crop(t, size_hw):
    H, W = t.shape[-2:]
    th, tw = size_hw
    dh, dw = (H-th)//2, (W-tw)//2
    return t[...,dh:dh+th, dw:dw+tw]

In [7]:
def fast_hist(pred, label, n_class=21):
    k = (0<=label) & (label<n_class)
    return np.bincount(n_class * label[k].astype(int)+pred[k].astype(int), minlength=n_class**2).reshape(n_class, n_class)

def miou_from_hist(hist):
    iu = np.diag(hist)/(hist.sum(1)+hist.sum(0)-np.diag(hist)+1e-10)
    return float(np.nanmean(iu)), iu

In [8]:
from tqdm import tqdm
def train(model, criterion, opt, scaler, loader, device, epochs):
    model.train()
    run = 0.0
    t0 = time.time()
    for x,y in tqdm(loader):
        imgs = x.to(device)
        masks = y.to(device)

        opt.zero_grad(set_to_none=True) # 메모리 절약
        # mac
        with torch.amp.autocast(device_type="mps", enabled=torch.backends.mps.is_available()):
        # cuda
        # with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            logits = model(imgs)
            masks_c = center_crop(masks, logits.shape[-2:])
            loss = criterion(logits, masks_c.long())
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        run += loss.item() * imgs.size(0)
    avg_loss = run / len(loader.dataset)
    print(f'[Epoch {epochs}] loss: {avg_loss:.3f} | time: {time.time()-t0:.3f} sec.')
    return avg_loss

@torch.no_grad()
def evaluate(model, loader, device, epoch):
    model.eval()
    hist = np.zeros((N_CLASSES,N_CLASSES),dtype=np.float64)
    for batch in  tqdm(loader):
        if len(batch) == 3:
            x,y,id_=batch
        else:
            x,y=batch
        imgs=x.to(device)
        masks=y.to(device)

        logits = model(imgs)
        masks_c=center_crop(masks,logits.shape[-2:])
        preds=torch.argmax(logits,dim=1)
        hist+= fast_hist(preds.cpu().numpy(),masks_c.cpu().numpy(),N_CLASSES)
    miou, iu = miou_from_hist(hist)
    print(f'[Epoch{epoch}] mIoU:{miou*100:.2f}%')
    return miou

In [9]:
# 데이터 준비
tr_ds, val_ds, tr_ds_loader, val_ds_loader = get_loaders()

In [10]:
# 모델 학습
m = UNet(3, N_CLASSES).to(DEVICE)
criterion = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)
opt = torch.optim.SGD(m.parameters(), lr=LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
# cuda
# scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())
# mac
scaler = torch.amp.GradScaler(enabled=torch.backends.mps.is_available())

In [11]:
#색 정립 (배경/무시 정리)

PAL = np.array([
    [0,0,0],[128,0,0],[0,128,0],[128,128,0],[0,0,128],
    [128,0,128],[0,128,128],[128,128,128],[64,0,0],[192,0,0],
    [64,128,0],[192,128,0],[64,0,128],[192,0,128],[64,128,128],
    [192,128,128],[0,64,0],[128,64,0],[0,192,0],[128,192,0],[0,64,128]
], dtype=np.uint8)
BG_COLOR=np.array([50,50,50],dtype=np.uint8)        #배경(0): 어두운 회색
IGNORE_COLOR=np.array([100,100,100],dtype=np.uint8) #무시(255): 밝은 회색

def _denorm(img):
    mean=torch.tensor([0.485, 0.456, 0.406],device=img.device)[:,None,None]
    std=torch.tensor([0.229, 0.224, 0.225],device=img.device)[:,None,None]
    return (img*std+mean).clamp(0,1)

def _colorize_mask(mask_np,ignore_idx):
    out=PAL[np.clip(mask_np,0,len(PAL)-1)].copy()
    out[mask_np == 0]=BG_COLOR
    out[mask_np == ignore_idx]=IGNORE_COLOR
    return out


@torch.no_grad()
def visualize_val_grid_random(epoch,model,val_ds,device,out_dir=OUT_DIR, samples=4):
    if len(val_ds)==0:
        print("검증 안함")
        return
    os.makedirs(out_dir,exist_ok=True)

    idxs = np.random.choice(len(val_ds),size=min(samples,len(val_ds)),replace=False)
    imgs,gts=[],[]
    for idx in idxs:
        img,mask,_ = val_ds[idx]
        imgs.append(img)
        gts.append(mask)
    batch = torch.stack(imgs).to(device)
    gts_np = [m.numpy() for m in gts]

    model.eval()
    out=model(batch)
    out=out['out'] if isinstance(out, dict) else out
    preds=out.argmax(1).detach().cpu().numpy()

    B=len(idxs)
    f,ax=plt.subplots(B,3,figsize=(12,3*B))

    if B==1:
        ax=np.expand_dims(ax,0)
    for i in range(B):
        img_np   = _denorm(batch[i]).detach().cpu().numpy().transpose(1, 2, 0)
        gt_rgb = _colorize_mask(gts_np[i],IGNORE_INDEX)
        pred_rgb = _colorize_mask(preds[i],IGNORE_INDEX)

        ax[i,0].imshow(img_np)
        ax[i,0].set_title("oj_img")
        ax[i,0].axis('off')

        ax[i,1].imshow(gt_rgb)
        ax[i,1].set_title("oj_mask")
        ax[i,1].axis('off')

        ax[i,2].imshow(pred_rgb)
        ax[i,2].set_title("pr_mask")
        ax[i,2].axis('off')
    plt.suptitle(f'val_(R)-Epoch{epoch}')
    save_path=os.path.join(out_dir,f'val_Epoch{epoch}.grid.png')
    plt.tight_layout()
    plt.savefig(save_path,dpi=150)
    plt.show()
    plt.close()
    print(f"기록 저장됨{save_path}")

In [ ]:
best_miou = -1.0
for epoch in range(1, EPOCHS+1):
    train(m, criterion, opt, scaler, tr_ds_loader, DEVICE, epoch)
    miou = evaluate(m, val_ds_loader, DEVICE, epoch)
    visualize_val_grid_random(epoch, m, val_ds, DEVICE, out_dir=OUT_DIR, samples=2)
    if miou > best_miou:
        best_miou = miou
        torch.save({
            'epoch':epoch,
            'model':m.state_dict(),
            'miou':best_miou,
            'config':{'IMG_SIZE':IMG_SIZE,
                      'N_CLASSES':N_CLASSES,
                      'LR':LR,
                      'MOMENTUM':MOMENTUM,
                      'WEIGHT_DECAY':WEIGHT_DECAY}
            },BEST_PATH)
        print(f'best_m mIou: {best_miou} 저장 진행')